<a href="https://colab.research.google.com/github/LeonardooAlves/LSE-ST445/blob/main/Week%205/1_st445_week5_lab1_sqlite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ST445 Week 5: relational databases with SQLite

**A self-guided notebook · follows Part 1 of the Week 5 lecture**

In this notebook you build a small relational database, step by step, with **SQLite** and Python. You will learn how to:

1. **create tables**, choosing a **data type** for every column;
2. set **primary keys** and **foreign keys**, and other **constraints** that stop bad data getting in;
3. carry out the four **CRUD** operations: **C**reate (`INSERT`), **R**ead (`SELECT`), **U**pdate (`UPDATE`) and **D**elete (`DELETE`);
4. **normalise** a messy table into first, second and third normal form.

The examples use the same country–year data as the lecture.

### How each piece of SQL is introduced

As in the lecture, every SQL statement appears in three steps:

| Step | What you see |
|---|---|
| **General form** | the statement's pattern, with placeholders such as `{table}` that you replace with your own names |
| **Example** | the statement written out for our data, and run |
| **Result** | what came back, explained in the Markdown cell underneath |

**Reading the general forms:** keywords such as `SELECT` are written in capitals (SQL does not require this, but it makes the structure easy to see); `{placeholder}` means "put your own name or value here"; `[ ... ]` means optional; text values go in single quotes, `'like this'`; `--` starts a comment; `;` ends a statement.

### Cell marks

As in our previous labs:
🟢 **run and read** · 🟡 **fill the gaps** marked `____` · 🔵 **write it yourself** · ✍️ **write a short answer**. Exercise cells are followed by a check cell that prints ✅ or ❌.

**Before you start:** **File → Save a copy in Drive**, so your work is kept.

---
## 0. Setting up

### What is SQLite?

**SQLite** is a relational database that lives in **a single file** on your computer. There is no server to install or start: Python's standard library includes a module called **`sqlite3`** that talks to it. That makes it ideal for learning, and it is also used in phones, browsers and many applications.

Python does not run SQL itself. Instead, it **sends** each SQL statement, as a piece of text, to the database through a **connection**, and collects the answer. The table below lists the Python methods you will use.

| Python | What it does |
|---|---|
| `con = sqlite3.connect("file.db")` | opens (or creates) the database file and returns a **connection** |
| `con.execute(sql)` | sends one SQL statement |
| `con.execute(sql, values)` | sends one statement, filling its `?` placeholders with `values` |
| `con.executemany(sql, rows)` | runs the same statement once for every row in `rows` |
| `cursor.fetchone()`, `cursor.fetchall()` | collects one result row, or all of them |
| `con.commit()` | makes your changes permanent in the file |
| `con.close()` | closes the connection |

In [1]:
# 🟢 Import the two modules we need
import sqlite3          # talks to SQLite databases; part of Python's standard library
import pandas as pd     # used only to display query results as tidy tables

print("SQLite version:", sqlite3.sqlite_version)

SQLite version: 3.45.1


**What you should see:** a version number such as `3.45.1`. Any version from 3.37 onwards supports everything in this notebook.

### Connect to a database file

`sqlite3.connect` opens the file if it exists and **creates it** if it does not. We first delete any old copy, so that you can re-run the notebook from the top and always start fresh.

In [2]:
# 🟢 Start with a fresh database file
from pathlib import Path

DB_FILE = Path("week5.db")

if DB_FILE.exists():
    DB_FILE.unlink()            # delete the old file, if there is one. This is used only for the purposes of running this lab always from a fresh start.

con = sqlite3.connect(DB_FILE)  # creates week5.db and opens a connection to it

con.execute("PRAGMA foreign_keys = ON")   # see the note below
print("Connected to", DB_FILE)

Connected to week5.db


**What happened:** a new, empty file `week5.db` now exists (open the **Files** panel on the left to see it), and `con` is our connection to it.

**About `PRAGMA foreign_keys = ON`.** A `PRAGMA` is an SQLite-specific setting. For historical reasons, SQLite does **not** check foreign keys unless you switch the check on, and the setting lasts only for the current connection. We switch it on straight away, every time we connect. (Most other databases check foreign keys by default.)

To read your database file, you can use [DB Browser for SQLite.](https://sqlitebrowser.org/)

### A helper to display results

[`pd.read_sql_query`](https://pandas.pydata.org/docs/reference/api/pandas.read_sql_query.html) runs a `SELECT` and returns the answer as a pandas DataFrame, which displays as a neat table. We wrap it in a small function, `show`, so each query needs only one line. Read the function: it does nothing you could not do yourself.

In [3]:
# 🟢 A helper: run a SELECT query and return the result as a table
def show(sql, values=()):
    """Run a SELECT statement and return its result as a pandas DataFrame."""
    result = pd.read_sql_query(sql, con, params=values)
    return result

def check(condition, hint="Not yet: read the task again."):
    """Print a tick if condition is True, otherwise a cross and a hint."""
    if condition:
        print("✅ Correct")
    else:
        print("❌", hint)

---
## 1. Creating tables, and choosing data types

### 1.1 The `CREATE TABLE` statement

A table has a **name** and a list of **columns**. Each column has a **name**, a **data type**, and optionally one or more **constraints** (rules/constraints, covered in section 2 of the lecture).

**General form**

```sql
CREATE TABLE {table} (
    {column} {type} [constraints],
    {column} {type} [constraints],
    ...
);
```

* The column definitions are inside round brackets and **separated by commas** (no comma after the last one).
* `CREATE TABLE IF NOT EXISTS {table} (...)` does nothing if the table already exists, instead of raising an error.

### 1.2 Data types in SQLite

SQLite stores every value in one of five **storage classes**:

| Type | Stores | Examples |
|---|---|---|
| `INTEGER` | whole numbers | `2022`, `-5`, `0` |
| `REAL` | decimal numbers (floating point, as in Week 1) | `48900.0`, `81.2` |
| `TEXT` | text (Unicode, as in Week 1) | `'GBR'`, `'United Kingdom'` |
| `BLOB` | raw bytes, stored exactly as given | an image, a file |
| `NULL` | a missing value | `NULL` |

Two things SQLite does **not** have as separate types, and how we store them:

* **Dates and times** are stored as **`TEXT` in ISO 8601** form, `'2026-10-05T14:30:00Z'` (Week 1): they then sort correctly as text, and SQLite's date functions understand them.
* **True/false values** are stored as **`INTEGER`**: `1` for true and `0` for false.

Other databases have more types (for example `DATE`, `BOOLEAN`, `VARCHAR(50)`, `NUMERIC(10, 2)` in PostgreSQL), but the idea is the same: **choose the type that matches what the column holds.**

### 1.3 Our first table

**Example.** A table of countries. We write the SQL as a multi-line Python string (between three double quote marks at each end) so it stays readable, then send it with `con.execute`.

In [4]:
# 🟢 Create a table with one column of each common type
create_sql = """
CREATE TABLE country_info (
    iso3          TEXT,       -- a three-letter code
    name          TEXT,       -- the country's name
    population_m  REAL,       -- population in millions: a decimal number
    joined_un     INTEGER,    -- the year it joined the United Nations: a whole number
    is_island     INTEGER,    -- true/false, stored as 1 or 0
    updated_at    TEXT        -- a timestamp, in ISO 8601 text
);
"""

con.execute(create_sql)
print("Table created")

Table created


**What happened:** SQLite created an empty table `country_info` with six columns. Nothing is printed by SQLite itself; our `print` confirms the statement ran without error.

Every database keeps a catalogue of its own tables. In SQLite it is a special table called `sqlite_schema`. Query it to confirm the table exists, and use `PRAGMA table_info` to list its columns:

In [5]:
# 🟢 Which tables exist, and what columns does ours have?
tables = pd.read_sql_query("SELECT name FROM sqlite_schema WHERE type = 'table'", con) #show("SELECT name FROM sqlite_schema WHERE type = 'table'")
print(tables)

columns = pd.read_sql_query("PRAGMA table_info(country_info)", con) #show("PRAGMA table_info(country_info)")
columns[["name", "type"]]

           name
0  country_info


,name,type
0,iso3,TEXT
1,name,TEXT
2,population_m,REAL
3,joined_un,INTEGER
4,is_island,INTEGER
5,updated_at,TEXT


**What you should see:** the tables in the database, in this case, `country_info`, and its six columns with the types we declared: `TEXT`, `TEXT`, `REAL`, `INTEGER`, `INTEGER`, `TEXT`.

### 1.4 What SQLite does with types: `typeof()`

SQLite is **flexibly typed**: by default it uses the declared type as a *preference* (called **type affinity**) and converts values where it can. The function `typeof(...)` shows how each value was actually stored. Insert two rows (the `INSERT` statement is explained in section 3) and look:

In [6]:
# 🟢 Insert two rows, then ask SQLite how it stored each value
con.execute("INSERT INTO country_info VALUES ('GBR', 'United Kingdom', 68.3, 1945, 1, '2026-10-05T14:30:00Z')")
con.execute("INSERT INTO country_info VALUES ('FRA', 'France', '68.2', '1945', 0, '2026-10-05T14:30:00Z')")

types_sql = pd.read_sql_query("""
SELECT iso3,
       population_m, typeof(population_m) AS stored_as,
       joined_un,    typeof(joined_un)    AS joined_stored_as
FROM country_info;
""", con)
types_sql
#show(types_sql)

,iso3,population_m,stored_as,joined_un,joined_stored_as
0,GBR,68.3,real,1945,integer
1,FRA,68.2,real,1945,integer


**What you should see:** for France we passed the population and the year as **text** (`'68.2'` and `'1945'`, in quotes), yet `typeof` reports `real` and `integer`: SQLite converted them to the declared types because they looked like numbers.

That flexibility is convenient, but it can hide mistakes: a value such as `'unknown'` in a `REAL` column would simply be stored as text. Since version 3.37, SQLite offers **`STRICT` tables**, which reject a value of the wrong type, as most other databases do:

In [7]:
# 🟢 A STRICT table rejects values of the wrong type
con.execute("""
CREATE TABLE strict_demo (
    year    INTEGER,
    value   REAL
) STRICT;
""")

try:
    con.execute("INSERT INTO strict_demo VALUES ('not a year', 1.5)")
except sqlite3.Error as err:
    print("Rejected:", type(err).__name__, "-", err)

Rejected: IntegrityError - cannot store TEXT value in INTEGER column strict_demo.year


**What you should see:** `Rejected: IntegrityError - cannot store TEXT value in INTEGER column strict_demo.year`.

**About `try` / `except`.** Some cells below are *meant* to fail, to show a rule in action. `try:` runs the indented code; if SQLite raises an error, `except ... as err:` catches it and we print the message instead of stopping the notebook.

We have finished with these two demonstration tables, so we remove them. `DROP TABLE` deletes a table and all its rows:

In [8]:
# 🟢 Remove the demonstration tables
con.execute("DROP TABLE country_info")
con.execute("DROP TABLE strict_demo")
con.commit()
print("Dropped")

Dropped


In [9]:
tables = pd.read_sql_query("SELECT name FROM sqlite_schema WHERE type = 'table'", con) #show("SELECT name FROM sqlite_schema WHERE type = 'table'")
print(tables)

Empty DataFrame
Columns: [name]
Index: []


### 🔵 Exercise 1: create a table

Create a table called `city` with these columns, choosing a sensible type for each: `city_id` (a whole number), `name` (text), `iso3` (text), `population_m` (a decimal number), `is_capital` (true/false). Then run the check.

In [ ]:
# Your code here

In [ ]:
# Check for exercise 1
info = show("PRAGMA table_info(city)")
types = dict(zip(info["name"], info["type"]))
check(types.get("city_id") == "INTEGER" and types.get("population_m") == "REAL" and types.get("is_capital") == "INTEGER",
      "city_id INTEGER, name TEXT, iso3 TEXT, population_m REAL, is_capital INTEGER.")

---
## 2. Primary keys, foreign keys and other constraints

A **constraint** is a rule the database checks every time data are written. If a new or changed row breaks a rule, the database **refuses** it. This is how a database protects **integrity**, one of the four reasons for using one (lecture, section 01 and 02).

| Constraint | Rule |
|---|---|
| `PRIMARY KEY` | identifies each row: the value must be **unique** and **not missing** |
| `NOT NULL` | a value is required |
| `UNIQUE` | no two rows may have the same value |
| `CHECK ({condition})` | every row must satisfy the condition |
| `DEFAULT {value}` | the value to use when none is given |
| `REFERENCES {table}({column})` | a **foreign key**: the value must exist in another table |

### 2.1 A primary key

**General form**: either after the column, for a single-column key

```sql
{column} {type} PRIMARY KEY
```

or as a separate line at the end, which allows a **composite key** of several columns:

```sql
PRIMARY KEY ({column1}, {column2})
```

**Example.** In our country table, the ISO3 code identifies each country, so it is the primary key.

In [10]:
# 🟢 The country table: iso3 is the primary key
country_sql = """
CREATE TABLE country (
    iso3          TEXT PRIMARY KEY,
    name          TEXT NOT NULL,
    income_group  TEXT NOT NULL
                  CHECK (income_group IN ('Low income', 'Lower middle income',
                                          'Upper middle income', 'High income')),
    created_at    TEXT DEFAULT (datetime('now'))
);
"""
con.execute(country_sql)
print("country created")

country created


**Reading the definition, column by column:**

* `iso3 TEXT PRIMARY KEY`: unique, and never missing.
* `name TEXT NOT NULL`: every country must have a name.
* `income_group ... CHECK (income_group IN (...))`: only the four World Bank labels are allowed. The constraint is written over three lines only to fit the page: SQL ignores line breaks.
* `created_at TEXT DEFAULT (datetime('now'))`: if we do not give a value, SQLite fills in the current date and time, in UTC.

You can check the metadata of your database to see what's in there.

In [11]:
# check all the tables in the database
tables = show("SELECT name FROM sqlite_schema WHERE type = 'table'")
tables

,name
0,country


In [12]:
columns = show("PRAGMA table_info(country)")
columns

,cid,name,type,notnull,dflt_value,pk
0,0,iso3,TEXT,0,None,1
1,1,name,TEXT,1,None,0
2,2,income_group,TEXT,1,None,0
3,3,created_at,TEXT,0,datetime('now'),0


**What you should see:** You can see the data type of each column. You can also see which columns do not accept empty/null values: `name` and `income_group`. You can also see that `pk` shows all columns considered a primary key, in this case `iso3`. `created_at` has a default value when adding a new room and this is not provided.



Remember that SQLite does not have a native DATETIME storage class, unlike other database management systems (like PostgreSQL, MySQL, or SQL Server).

Therefore, SQLite allows you to store dates and times using three options:

- `TEXT` as ISO 8601 strings (e.g., '2026-10-28 14:16:52').
- `REAL` as Julian day numbers (the number of days since noon in Greenwich on November 24, 4714 B.C.).
- `INTEGER` as Unix Time (the number of seconds since 1970-01-01 00:00:00 UTC).

TEXT is preferred (ISO 8601) due to two major advantages:

- Human Readability: You can run `SELECT * FROM country` and instantly read the creation date without needing to write code to convert it.
- Lexicographical Sorting: Because ISO 8601 format is structured from most-significant to least-significant unit (YYYY-MM-DD HH:MM:SS), sorting alphabetically matches sorting chronologically.

### 2.2 A composite key and a foreign key

The indicator table holds one value **per country and year**, so its primary key is the **pair** `(iso3, year)`: a **composite key**. Its `iso3` column must also refer to a country that exists: a **foreign key**.

**General form of a foreign key**

```sql
{column} {type} REFERENCES {other_table}({other_column})
-- or, as a separate line:
FOREIGN KEY ({column}) REFERENCES {other_table}({other_column})
```

In [13]:
# 🟢 The indicator table: a composite primary key, and a foreign key to country
indicator_sql = """
CREATE TABLE indicator (
    iso3             TEXT    NOT NULL,
    year             INTEGER NOT NULL CHECK (year BETWEEN 1960 AND 2030),
    gdp_pc_usd       REAL             CHECK (gdp_pc_usd > 0),
    life_expectancy  REAL             CHECK (life_expectancy BETWEEN 20 AND 100),
    PRIMARY KEY (iso3, year),
    FOREIGN KEY (iso3) REFERENCES country(iso3)
);
"""
con.execute(indicator_sql)
print("indicator created")

indicator created


**What this says:** each `(iso3, year)` pair appears at most once; `iso3` must already exist in `country`; the year, GDP and life expectancy must be in plausible ranges. These are the Week 4 validation checks, now enforced **by the database itself**, on every write.

Notice that `gdp_pc_usd` and `life_expectancy` have no `NOT NULL`: a missing value is allowed (it is stored as `NULL`), but a value that is present must pass the `CHECK`.

In [14]:
columns = show("PRAGMA table_info(indicator)")
columns

,cid,name,type,notnull,dflt_value,pk
0,0,iso3,TEXT,1,None,1
1,1,year,INTEGER,1,None,2
2,2,gdp_pc_usd,REAL,0,None,0
3,3,life_expectancy,REAL,0,None,0


### 2.3 Watching the constraints work

Add some valid rows first. `executemany` runs the same `INSERT` once for each tuple in the list; the `?` marks are **placeholders** that are filled in with the values, in order (explained fully in section 3).

In [15]:
# 🟢 Valid data: five countries, seven indicator rows
countries = [
    ("GBR", "United Kingdom", "High income"),
    ("FRA", "France", "High income"),
    ("IND", "India", "Lower middle income"),
    ("NGA", "Nigeria", "Lower middle income"),
    ("QAT", "Qatar", "High income"),
]
con.executemany("INSERT INTO country (iso3, name, income_group) VALUES (?, ?, ?)", countries)

indicators = [
    ("GBR", 2021, 46500, 80.7),
    ("GBR", 2022, 48900, 80.9),
    ("FRA", 2021, 43700, 82.3),
    ("FRA", 2022, 44400, 82.5),
    ("IND", 2021, 2250, 67.2),
    ("IND", 2022, 2410, 67.7),
    ("NGA", 2022, 2160, 53.6),
]
con.executemany("INSERT INTO indicator VALUES (?, ?, ?, ?)", indicators)

con.commit()
show("SELECT * FROM country")

,iso3,name,income_group,created_at
0,GBR,United Kingdom,High income,2026-10-29 10:22:58
1,FRA,France,High income,2026-10-29 10:22:58
2,IND,India,Lower middle income,2026-10-29 10:22:58
3,NGA,Nigeria,Lower middle income,2026-10-29 10:22:58
4,QAT,Qatar,High income,2026-10-29 10:22:58


**What you should see:** five countries. The `created_at` column was filled automatically by the `DEFAULT`, because our `INSERT` named only three columns.

In [16]:
# You can also visualise the indicator table
show("SELECT * FROM indicator")

,iso3,year,gdp_pc_usd,life_expectancy
0,GBR,2021,46500.0,80.7
1,GBR,2022,48900.0,80.9
2,FRA,2021,43700.0,82.3
3,FRA,2022,44400.0,82.5
4,IND,2021,2250.0,67.2
5,IND,2022,2410.0,67.7
6,NGA,2022,2160.0,53.6


*NOTE: SQLite lets you see your own uncommitted changes. If you insert a row in cell A, and query it in cell B using the same `con` connection, the row will appear even if you haven't committed. However, if you close the notebook, lose connection, or try to open week5.db using an external program (like DB Browser), those changes will not be saved and will be lost forever unless `con.commit()` was called.*



Now try to break each rule. Every statement below should be **rejected**:

In [17]:
# 🟢 Four statements that break a constraint
bad_statements = [
    ("a country code that does not exist",  "INSERT INTO indicator VALUES ('XXX', 2022, 1000, 60)"),
    ("the same country-year twice",          "INSERT INTO indicator VALUES ('GBR', 2022, 50000, 81)"),
    ("a negative GDP",                       "INSERT INTO indicator VALUES ('GBR', 2023, -5, 81)"),
    ("an income group that is not allowed",  "INSERT INTO country (iso3, name, income_group) VALUES ('ESP', 'Spain', 'low income')"),
]

for description, sql in bad_statements:
    try:
        con.execute(sql)
        print("Accepted (unexpected):", description)
    except sqlite3.IntegrityError as err:
        print(f"Rejected {description}: {err}")

Rejected a country code that does not exist: FOREIGN KEY constraint failed
Rejected the same country-year twice: UNIQUE constraint failed: indicator.iso3, indicator.year
Rejected a negative GDP: CHECK constraint failed: gdp_pc_usd > 0
Rejected an income group that is not allowed: CHECK constraint failed: income_group IN ('Low income', 'Lower middle income',
                                          'Upper middle income', 'High income')


**What you should see:** four rejections, each with the rule that was broken:

* `FOREIGN KEY constraint failed`: there is no country `XXX` in `country.iso3`;
* `UNIQUE constraint failed: indicator.iso3, indicator.year`: the composite primary key already has `('GBR', 2022)`;
* `CHECK constraint failed: gdp_pc_usd > 0`;
* `CHECK constraint failed: income_group IN (...)`.

**The loop, line by line:** `for description, sql in bad_statements:` takes each pair from the list in turn; `try` runs the statement; if SQLite refuses it, `except` catches the `IntegrityError` and prints its message.

Foreign keys also protect the other direction: you cannot **delete** a country that indicator rows still refer to, because that would leave them pointing at nothing.

In [18]:
# 🟢 Deleting a country that is still referred to
try:
    con.execute("DELETE FROM country WHERE iso3 = 'GBR'")
except sqlite3.IntegrityError as err:
    print("Rejected:", err)

Rejected: FOREIGN KEY constraint failed


**What you should see:** `Rejected: FOREIGN KEY constraint failed`. This is **referential integrity**: every foreign key value always refers to a row that exists.

### 🟡 Exercise 2: a table with keys

Complete the definition of a `city` table (we drop the exercise 1 version first) in which: `city_id` is the primary key; `name` is required; `iso3` is a foreign key to `country`; and `population_m` must be positive. Replace every `____`.

In [ ]:
con.execute("DROP TABLE IF EXISTS city")

city_sql = """
CREATE TABLE city (
    city_id       INTEGER ____,
    name          TEXT    ____,
    iso3          TEXT    NOT NULL ____ country(iso3),
    population_m  REAL    CHECK (____)
);
"""
con.execute(city_sql)
print("Table city created")

In [ ]:
# Check for exercise 2: a valid city is accepted; three invalid ones are rejected
con.execute("DELETE FROM city")
con.execute("INSERT INTO city VALUES (1, 'London', 'GBR', 9.0)")
rejected = 0
for sql in ["INSERT INTO city VALUES (1, 'Paris', 'FRA', 2.1)",      # duplicate primary key
            "INSERT INTO city VALUES (2, 'Atlantis', 'XXX', 1.0)",   # unknown country
            "INSERT INTO city VALUES (3, 'Ghost', 'FRA', -1)"]:      # negative population
    try:
        con.execute(sql)
    except sqlite3.IntegrityError:
        rejected = rejected + 1
check(rejected == 3, f"{rejected} of 3 invalid rows were rejected: check PRIMARY KEY, REFERENCES and CHECK.")
con.commit()

---
## 3. CRUD: create, read, update, delete

| Operation | SQL statement | General form |
|---|---|---|
| **C**reate | `INSERT` | `INSERT INTO {table} ({columns}) VALUES ({values});` |
| **R**ead | `SELECT` | `SELECT {columns} FROM {table} [WHERE {condition}];` |
| **U**pdate | `UPDATE` | `UPDATE {table} SET {column} = {value} WHERE {condition};` |
| **D**elete | `DELETE` | `DELETE FROM {table} WHERE {condition};` |

### 3.1 Create: `INSERT`

**General form**

```sql
INSERT INTO {table} ({column1}, {column2}, ...)
VALUES ({value1}, {value2}, ...);
```

The values are matched to the columns **in order**. Naming the columns is optional if you give a value for every column in the table's order, but naming them is safer: the statement still works if the table later gains a column.

**Values from Python: always use placeholders.** Each `?` in the statement is filled, in order, from a tuple of values passed separately. Never paste values into the SQL text yourself (section 3.6 shows why).

In [19]:
# 🟢 Insert one row, with placeholders
sql = "INSERT INTO country (iso3, name, income_group) VALUES (?, ?, ?)"
values = ("ESP", "Spain", "High income")

con.execute(sql, values)
con.commit()

show("SELECT iso3, name, income_group FROM country WHERE iso3 = 'ESP'")

,iso3,name,income_group
0,ESP,Spain,High income


**What happened:** the three `?` marks were filled with `"ESP"`, `"Spain"` and `"High income"`, in that order, and the row was added. `con.commit()` saved it to the file.

**Inserting many rows** uses `executemany` with a **list of tuples**, one tuple per row. Section 2 used it already.

### 3.2 Read: `SELECT`, one clause at a time

A `SELECT` statement is built from **clauses**. We add them one at a time and look at the result after each.

**General form**

```sql
SELECT {columns}
FROM {table}
[WHERE {condition}]
[ORDER BY {column} [DESC]]
[LIMIT {n}];
```

**Step 1: which columns, from which table.** `*` means every column.

In [20]:
# 🟢 Step 1: two columns from one table
show("SELECT iso3, name FROM country")

,iso3,name
0,GBR,United Kingdom
1,FRA,France
2,IND,India
3,NGA,Nigeria
4,QAT,Qatar
5,ESP,Spain


**What you should see:** six rows (the five countries plus Spain), with only the two columns we asked for.

**Step 2: `WHERE` keeps only the rows that meet a condition.**

In [21]:
# 🟢 Step 2: only high-income countries
query = """
SELECT iso3, name
FROM country
WHERE income_group = 'High income';
"""
show(query)

,iso3,name
0,GBR,United Kingdom
1,FRA,France
2,QAT,Qatar
3,ESP,Spain


**What you should see:** four rows: United Kingdom, France, Qatar and Spain. Text values in SQL use **single quotes**.

**Step 3: `ORDER BY` sorts; `LIMIT` keeps the first rows.** Without `ORDER BY`, the order of rows is **not guaranteed**: a table is a set of rows, with no order of its own (lecture, the relational model).

In [22]:
# 🟢 Step 3: sorted by name, first two only
query = """
SELECT iso3, name
FROM country
WHERE income_group = 'High income'
ORDER BY name
LIMIT 2;
"""
show(query)

,iso3,name
0,FRA,France
1,QAT,Qatar


**What you should see:** France and Qatar: the first two high-income countries in alphabetical order. Add `DESC` after `name` to sort from Z to A.

### 3.3 Writing conditions in `WHERE`

| Condition | Meaning |
|---|---|
| `{column} = {value}` | equal (one `=`, not two) |
| `{column} <> {value}` | not equal |
| `{column} > {value}` | greater than (also `<`, `>=`, `<=`) |
| `{column} BETWEEN {a} AND {b}` | from `a` to `b`, both included |
| `{column} IN ({v1}, {v2})` | equal to any value in the list |
| `{column} LIKE 'U%'` | text pattern: `%` matches any text, `_` one character |
| `{column} IS NULL` | the value is missing (never `= NULL`) |
| `{cond1} AND {cond2}` | both are true (also `OR`, and `NOT`) |

In [23]:
# 🟢 Two conditions combined with AND
query = """
SELECT iso3, year, gdp_pc_usd
FROM indicator
WHERE year = 2022
  AND gdp_pc_usd > 10000;
"""
show(query)

,iso3,year,gdp_pc_usd
0,GBR,2022,48900.0
1,FRA,2022,44400.0


**What you should see:** GBR and FRA in 2022: the only rows meeting **both** conditions.

In [ ]:
# 🟢 A text pattern with LIKE, OR, and a list with IN
query = """
SELECT name
FROM country
WHERE name LIKE 'U%'
   OR iso3 IN ('IND', 'NGA');
"""
show(query)

**What you should see:** United Kingdom (its name starts with U), India and Nigeria (their codes are in the list).

**Missing values need `IS NULL`.** Add a row with no life expectancy, then compare the two ways of looking for it:

In [24]:
# 🟢 IS NULL versus = NULL
con.execute("INSERT INTO indicator (iso3, year, gdp_pc_usd) VALUES ('QAT', 2022, 87000)")
con.commit()

with_equals = show("SELECT iso3, year FROM indicator WHERE life_expectancy = NULL")
with_is_null = show("SELECT iso3, year FROM indicator WHERE life_expectancy IS NULL")

print("Rows found with = NULL: ", len(with_equals))
print("Rows found with IS NULL:", len(with_is_null))

Rows found with = NULL:  0
Rows found with IS NULL: 1


**What you should see:** `0` with `= NULL` and `1` with `IS NULL`. In SQL, `NULL` means "unknown", and comparing anything with an unknown gives *unknown*, never true: the same idea as `NaN != NaN` in Week 3.

### 3.4 Summarising: `GROUP BY` and `HAVING`

**General form**

```sql
SELECT {group_column}, {aggregate}({column}) AS {name}
FROM {table}
[WHERE {row_condition}]
GROUP BY {group_column}
[HAVING {group_condition}];
```

* `GROUP BY` makes **one result row per group**.
* An **aggregate** summarises each group: [`COUNT(*)`](https://www.w3schools.com/sql/sql_count.asp) (number of rows), `SUM`, `AVG`, `MIN`, `MAX`.
* `AS {name}` gives a result column a name.
* `WHERE` filters **rows before** grouping; `HAVING` filters **groups after** grouping.

In [25]:
# 🟢 One row per country: how many years, and the average GDP per person
query = """
SELECT iso3,
       COUNT(*)              AS n_years, -- Because each row in the indicator table represents data for a single year for a country, counting the rows per country gives the total number of years that country has data for
       ROUND(AVG(gdp_pc_usd)) AS mean_gdp
FROM indicator
GROUP BY iso3;
"""
show(query)

,iso3,n_years,mean_gdp
0,FRA,2,44050.0
1,GBR,2,47700.0
2,IND,2,2330.0
3,NGA,1,2160.0
4,QAT,1,87000.0


**What you should see:** five rows, one per country: FRA, GBR and IND have 2 years; NGA and QAT have 1.

Now keep only the countries with two years. That is a condition on a **group** (its count), so it goes in `HAVING`, not `WHERE`:

In [26]:
# 🟢 HAVING filters the groups
query = """
SELECT iso3,
       COUNT(*)              AS n_years,
       ROUND(AVG(gdp_pc_usd)) AS mean_gdp
FROM indicator
GROUP BY iso3
HAVING COUNT(*) = 2;
"""
show(query)

,iso3,n_years,mean_gdp
0,FRA,2,44050.0
1,GBR,2,47700.0
2,IND,2,2330.0


**What you should see:** FRA, GBR and IND. NGA and QAT are dropped.

**Why can `WHERE` not use `COUNT(*)`?** Because SQL **evaluates** a query in a different order from the one you write it in: `FROM` → `WHERE` → `GROUP BY` → `HAVING` → `SELECT` → `ORDER BY`. When `WHERE` runs, the groups do not exist yet.

### 3.5 Reading results in Python, without pandas

`show` uses pandas for display. Underneath, `con.execute` returns a **cursor**, from which you collect rows as Python **tuples**:

In [27]:
# 🟢 Collect results as tuples
cursor = con.execute("SELECT iso3, name FROM country WHERE iso3 = ?", ("GBR",))
one_row = cursor.fetchone()
print("fetchone():", one_row)

cursor = con.execute("SELECT iso3, name FROM country ORDER BY name")
all_rows = cursor.fetchall()
print("fetchall():", all_rows)

first_name = all_rows[0][1]       # row 0, column 1
print("The first country's name:", first_name)

fetchone(): ('GBR', 'United Kingdom')
fetchall(): [('FRA', 'France'), ('IND', 'India'), ('NGA', 'Nigeria'), ('QAT', 'Qatar'), ('ESP', 'Spain'), ('GBR', 'United Kingdom')]
The first country's name: France


**What you should see:** `fetchone()` returns one tuple, `('GBR', 'United Kingdom')`; `fetchall()` returns a list of tuples, one per row. `all_rows[0][1]` takes the first row, then its second value (Python counts from 0): `'France'`, the first country in alphabetical order.

Note the comma in `("GBR",)`: a tuple with one item needs it, otherwise Python reads the brackets as ordinary brackets.

### 3.6 Why placeholders matter: SQL injection

Suppose the country code comes from a user. If you paste it into the SQL text with an f-string, a crafted input can **change the query**:

In [28]:
# 🟢 An f-string lets the input become part of the SQL
user_input = "x' OR '1'='1"

unsafe_sql = f"SELECT * FROM country WHERE iso3 = '{user_input}'"
print("The SQL that was sent:", unsafe_sql)
unsafe_rows = con.execute(unsafe_sql).fetchall()
print("Rows returned (unsafe):", len(unsafe_rows))

safe_rows = con.execute("SELECT * FROM country WHERE iso3 = ?", (user_input,)).fetchall()
print("Rows returned (safe):  ", len(safe_rows))

The SQL that was sent: SELECT * FROM country WHERE iso3 = 'x' OR '1'='1'
Rows returned (unsafe): 6
Rows returned (safe):   0


**What you should see:** the unsafe query becomes `... WHERE iso3 = 'x' OR '1'='1'`, which is true for every row, so it returns **all 6 countries**. With a placeholder, the input is treated purely as a value: no country has the code `x' OR '1'='1`, so **0 rows**. This is **SQL injection**; placeholders are the defence. **Never build SQL from values with f-strings or `+`.**

### 3.7 Update: `UPDATE`

**General form**

```sql
UPDATE {table}
SET {column} = {value}[, {column} = {value}, ...]
WHERE {condition};
```

`UPDATE` changes **every row that matches `WHERE`**, and every row in the table if you leave `WHERE` out. A safe routine has three steps:

1. run the same `WHERE` as a `SELECT`, to see which rows will change;
2. run the `UPDATE`;
3. check how many rows changed: [`cursor.rowcount`](https://docs.python.org/3/library/sqlite3.html#sqlite3.Cursor.rowcount).

In [29]:
# 🟢 Step 1: look first, with the same WHERE
show("SELECT * FROM indicator WHERE iso3 = 'GBR' AND year = 2022")

,iso3,year,gdp_pc_usd,life_expectancy
0,GBR,2022,48900.0,80.9


In [30]:
# 🟢 Steps 2 and 3: update, then check how many rows changed
sql = "UPDATE indicator SET life_expectancy = ? WHERE iso3 = ? AND year = ?"
values = (81.0, "GBR", 2022)

cursor = con.execute(sql, values)
print("Rows changed:", cursor.rowcount)

con.commit()
show("SELECT * FROM indicator WHERE iso3 = 'GBR' AND year = 2022")

Rows changed: 1


,iso3,year,gdp_pc_usd,life_expectancy
0,GBR,2022,48900.0,81.0


**What you should see:** one row before the update, `Rows changed: 1`, and the new life expectancy of 81.0 afterwards.

**Constraints still apply to updates.** Setting a negative GDP is rejected by the `CHECK`, exactly as an insert would be.

### 3.8 Delete: `DELETE`

**General form**

```sql
DELETE FROM {table}
WHERE {condition};
```

Without `WHERE`, **every row** is deleted. Use the same look-first routine as for `UPDATE`.

In [31]:
# 🟢 Delete Spain (it has no indicator rows, so the foreign key allows it)
cursor = con.execute("DELETE FROM country WHERE iso3 = ?", ("ESP",))
print("Rows deleted:", cursor.rowcount)

con.commit()
show("SELECT iso3 FROM country")

Rows deleted: 1


,iso3
0,FRA
1,GBR
2,IND
3,NGA
4,QAT


**What you should see:** `Rows deleted: 1`, and five countries left.

### 3.9 Transactions: all or nothing

A **transaction** groups several statements so that they are applied **completely or not at all**. This is the A of **ACID** (atomic, consistent, isolated, durable).

**General form in SQL**

```sql
BEGIN;
    {statement};
    {statement};
COMMIT;      -- apply all of them; or ROLLBACK; to undo all of them
```

**In Python**, the easiest way is a `with con:` block: if every statement inside succeeds, the changes are committed at the end; if any statement fails, **all** of them are rolled back.

**Example.** We try to replace Nigeria's 2022 row: delete the old one, then insert the new one. The new one has a negative GDP by mistake, so the insert fails, and the delete is undone too.

In [32]:
# 🟢 A transaction that fails part-way, and is rolled back
try:
    with con:
        con.execute("DELETE FROM indicator WHERE iso3 = 'NGA' AND year = 2022")
        con.execute("INSERT INTO indicator VALUES ('NGA', 2022, -1, 53.6)")
except sqlite3.IntegrityError as err:
    print("Rolled back:", err)

show("SELECT * FROM indicator WHERE iso3 = 'NGA'")

Rolled back: CHECK constraint failed: gdp_pc_usd > 0


,iso3,year,gdp_pc_usd,life_expectancy
0,NGA,2022,2160.0,53.6


**What you should see:** `Rolled back: CHECK constraint failed: gdp_pc_usd > 0`, and Nigeria's original row is **still there**. Without the transaction, the delete would have succeeded and the insert failed, leaving Nigeria with no data at all.

### 🟡 Exercise 3: CRUD

Complete each statement, then run the check. Use `?` placeholders for the values.

1. **Create:** add India's 2023 row: GDP per person 2480, life expectancy 68.0.
2. **Read:** select the `iso3` and `gdp_pc_usd` of every row from 2022, highest GDP first.
3. **Update:** change France's 2022 GDP per person to 44500, and store the number of rows changed in `n_changed`.
4. **Delete:** delete every row from 2021, and store the number deleted in `n_deleted`.

In [ ]:
# 1. Create
con.execute("INSERT INTO indicator VALUES (?, ?, ?, ?)", (____, ____, ____, ____))

# 2. Read
top_2022 = show("""
SELECT iso3, gdp_pc_usd
FROM indicator
WHERE ____
ORDER BY ____ ____;
""")
print(top_2022)

# 3. Update
cursor = con.execute("UPDATE indicator SET ____ = ? WHERE iso3 = ? AND year = ?", (____, ____, ____))
n_changed = cursor.____

# 4. Delete
cursor = con.execute("DELETE FROM indicator WHERE ____ = ?", (____,))
n_deleted = cursor.rowcount

con.commit()

In [ ]:
# Check for exercise 3
check(show("SELECT COUNT(*) AS n FROM indicator WHERE iso3 = 'IND' AND year = 2023")["n"][0] == 1, "1. India 2023 should exist.")
check(list(top_2022["iso3"]) == ["QAT", "GBR", "FRA", "IND", "NGA"], "2. WHERE year = 2022, ORDER BY gdp_pc_usd DESC.")
check(n_changed == 1 and show("SELECT gdp_pc_usd FROM indicator WHERE iso3 = 'FRA' AND year = 2022")["gdp_pc_usd"][0] == 44500,
      "3. SET gdp_pc_usd = 44500 for FRA, 2022; n_changed = cursor.rowcount.")
check(n_deleted == 3 and show("SELECT COUNT(*) AS n FROM indicator WHERE year = 2021")["n"][0] == 0, "4. DELETE ... WHERE year = 2021 (3 rows).")

---
## 4. Normalising a table

**Normalisation** reorganises tables so that **each fact is stored once**. It removes the redundancy that causes **anomalies**: errors that appear when data are inserted, updated or deleted. Each **normal form** removes one kind of redundancy:

| Normal form | Rule |
|---|---|
| **1NF** | every value is **atomic** (one value per cell); there are no repeating groups; every row is identified by a key |
| **2NF** | 1NF, and every non-key column depends on the **whole** key, not just part of a composite key |
| **3NF** | 2NF, and no non-key column depends on **another non-key column** |

Kent's summary (1983): every non-key column must depend on **the key, the whole key, and nothing but the key**.

### 4.1 The starting point: one big table

This is the kind of table you get when a spreadsheet is copied straight into a database. Its key is `(iso3, year)`.

In [33]:
# 🟢 A denormalised table, loaded from a list of tuples
con.execute("""
CREATE TABLE survey_raw (
    iso3                TEXT,
    year                INTEGER,
    country_name        TEXT,
    income_group_code   TEXT,
    income_group_label  TEXT,
    languages           TEXT,
    gdp_pc_usd          REAL,
    PRIMARY KEY (iso3, year)
);
""")

raw_rows = [
    ("GBR", 2021, "United Kingdom", "HIC", "High income",         "English; Welsh",       46500),
    ("GBR", 2022, "United Kingdom", "HIC", "High income",         "English; Welsh",       48900),
    ("FRA", 2021, "France",         "HIC", "High income",         "French",               43700),
    ("FRA", 2022, "France",         "HIC", "High income",         "French",               44400),
    ("IND", 2021, "India",          "LMC", "Lower middle income", "Hindi; English",       2250),
    ("IND", 2022, "India",          "LMC", "Lower middle income", "Hindi; English",       2410),
    ("NGA", 2022, "Nigeria",        "LMC", "Lower middle income", "English; Hausa; Yoruba", 2160),
]
con.executemany("INSERT INTO survey_raw VALUES (?, ?, ?, ?, ?, ?, ?)", raw_rows)
con.commit()

show("SELECT * FROM survey_raw")

,iso3,year,country_name,income_group_code,income_group_label,languages,gdp_pc_usd
0,GBR,2021,United Kingdom,HIC,High income,English; Welsh,46500.0
1,GBR,2022,United Kingdom,HIC,High income,English; Welsh,48900.0
2,FRA,2021,France,HIC,High income,French,43700.0
3,FRA,2022,France,HIC,High income,French,44400.0
4,IND,2021,India,LMC,Lower middle income,Hindi; English,2250.0
5,IND,2022,India,LMC,Lower middle income,Hindi; English,2410.0
6,NGA,2022,Nigeria,LMC,Lower middle income,English; Hausa; Yoruba,2160.0


**What you should see:** seven rows. Look at how often the same facts repeat: "United Kingdom", "HIC" and "High income" appear in every UK row; the languages are stored again for every year; and some cells hold **several** languages separated by semicolons.

### 4.2 The problems this causes: anomalies

**An update anomaly.** Suppose the UK's name is changed in one row but not the other:

In [34]:
# 🟢 Change the name in ONE of the two UK rows
con.execute("""
UPDATE survey_raw
SET country_name = 'United Kingdom of Great Britain and Northern Ireland'
WHERE iso3 = 'GBR' AND year = 2022;
""")

show("SELECT DISTINCT iso3, country_name FROM survey_raw WHERE iso3 = 'GBR'")

,iso3,country_name
0,GBR,United Kingdom
1,GBR,United Kingdom of Great Britain and Northern I...


**What you should see:** **two different names** for the same country. The table now contradicts itself, and nothing stopped it. (`SELECT DISTINCT` returns each different combination only once.)

The other two anomalies:

* **Insertion anomaly:** you cannot add a new country, say Spain, until it has a GDP value for some year, because every row needs a year: the key is `(iso3, year)`.
* **Deletion anomaly:** delete Nigeria's only row and you also lose its name, income group and languages.

We undo the inconsistent update before continuing:

In [35]:
# 🟢 Put the name back
con.execute("UPDATE survey_raw SET country_name = 'United Kingdom' WHERE iso3 = 'GBR'")
con.commit()

### 4.3 First normal form: one value per cell

`languages` holds a **list** in one cell ("English; Welsh"). That breaks 1NF: you cannot easily ask "which countries list English?", and a typo in one list is invisible.

**The fix:** a separate table with **one row per country and language**. Its key is the pair `(iso3, language)`.

The languages are stored once per country, not once per year, so we take each country's list once, split it at the semicolons in Python, and insert one row per language. The loop is written out step by step:

In [36]:
# 🟢 1NF: one row per country and language
con.execute("""
CREATE TABLE country_language (
    iso3      TEXT NOT NULL,
    language  TEXT NOT NULL,
    PRIMARY KEY (iso3, language)
);
""")

# one row per country, with its list of languages
lists = con.execute("SELECT DISTINCT iso3, languages FROM survey_raw").fetchall()

for iso3, languages in lists:
    parts = languages.split(";")          # "English; Welsh" -> ["English", " Welsh"]
    for part in parts:
        language = part.strip()           # remove the space left after the semicolon
        con.execute("INSERT INTO country_language VALUES (?, ?)", (iso3, language))

con.commit()
show("SELECT * FROM country_language ORDER BY iso3, language")

,iso3,language
0,FRA,French
1,GBR,English
2,GBR,Welsh
3,IND,English
4,IND,Hindi
5,NGA,English
6,NGA,Hausa
7,NGA,Yoruba


**What you should see:** eight rows, one per country and language: FRA French; GBR English and Welsh; IND English and Hindi; NGA English, Hausa and Yoruba.

**How the loop works:** `fetchall()` gives four `(iso3, languages)` pairs. For each, `split(";")` breaks the text at every semicolon into a list; `strip()` removes the leading space from each piece; each language is inserted as its own row.

Now the question that was awkward before is a simple `WHERE`:

In [37]:
# 🟢 Which countries list English?
show("SELECT iso3 FROM country_language WHERE language = 'English'")

,iso3
0,GBR
1,IND
2,NGA


**What you should see:** GBR, IND and NGA.

### 4.4 Second normal form: depend on the whole key

The key of `survey_raw` is `(iso3, year)`. Ask of each other column: does it depend on the **whole** key, or only on part of it?

| Column | Depends on | Verdict |
|---|---|---|
| `gdp_pc_usd` | the country **and** the year | the whole key: stays |
| `country_name` | the country only (`iso3`) | **part** of the key: breaks 2NF |
| `income_group_code`, `income_group_label` | the country only | **part** of the key: breaks 2NF |

**The fix:** move the columns that depend only on `iso3` to a table whose key **is** `iso3`. We build it with `INSERT INTO ... SELECT DISTINCT`, which copies the **results of a query** into a table.

**General form**

```sql
INSERT INTO {new_table} ({columns})
SELECT DISTINCT {columns} FROM {old_table};
```

In [38]:
# 🟢 2NF: country facts in their own table (still with the label: see 3NF)
con.execute("""
CREATE TABLE country_2nf (
    iso3                TEXT PRIMARY KEY,
    country_name        TEXT NOT NULL,
    income_group_code   TEXT NOT NULL,
    income_group_label  TEXT NOT NULL
);
""")

con.execute("""
INSERT INTO country_2nf (iso3, country_name, income_group_code, income_group_label)
SELECT DISTINCT iso3, country_name, income_group_code, income_group_label
FROM survey_raw;
""")
con.commit()

show("SELECT * FROM country_2nf")

,iso3,country_name,income_group_code,income_group_label
0,FRA,France,HIC,High income
1,GBR,United Kingdom,HIC,High income
2,IND,India,LMC,Lower middle income
3,NGA,Nigeria,LMC,Lower middle income


**What you should see:** **four** rows, one per country, instead of seven. `DISTINCT` removed the repetition across years. Each country's name is now stored **once**, so the update anomaly in 4.2 can no longer happen.

### 4.5 Third normal form: nothing but the key

In `country_2nf`, the key is `iso3`. But `income_group_label` does not really depend on the country: it depends on `income_group_code`. Every row with `HIC` has "High income", every row with `LMC` has "Lower middle income". That is a dependency between **two non-key columns** (a *transitive* dependency: `iso3` → code → label), and it breaks 3NF.

**The fix:** a small lookup table for income groups, and only the code in the country table, as a foreign key.

In [39]:
# 🟢 3NF: an income-group lookup table
con.execute("""
CREATE TABLE income_group (
    code   TEXT PRIMARY KEY,
    label  TEXT NOT NULL UNIQUE
);
""")

con.execute("""
INSERT INTO income_group (code, label)
SELECT DISTINCT income_group_code, income_group_label
FROM country_2nf;
""")

show("SELECT * FROM income_group")

,code,label
0,HIC,High income
1,LMC,Lower middle income


**What you should see:** two rows: `HIC` High income, `LMC` Lower middle income. Each label is stored once.

Now the final tables. Each has a primary key, and the foreign keys link them:

```text
income_group (code)  ←  country (iso3, income_group_code)  ←  measure (iso3, year)
                                    ↑
                         country_language (iso3, language)
```

In [40]:
# 🟢 The final, normalised tables
con.execute("""
CREATE TABLE country_3nf (
    iso3               TEXT PRIMARY KEY,
    name               TEXT NOT NULL,
    income_group_code  TEXT NOT NULL REFERENCES income_group(code)
);
""")
con.execute("""
INSERT INTO country_3nf (iso3, name, income_group_code)
SELECT iso3, country_name, income_group_code
FROM country_2nf;
""")

con.execute("""
CREATE TABLE measure (
    iso3        TEXT    NOT NULL REFERENCES country_3nf(iso3),
    year        INTEGER NOT NULL,
    gdp_pc_usd  REAL    CHECK (gdp_pc_usd > 0),
    PRIMARY KEY (iso3, year)
);
""")
con.execute("""
INSERT INTO measure (iso3, year, gdp_pc_usd)
SELECT iso3, year, gdp_pc_usd
FROM survey_raw;
""")
con.commit()

show("SELECT * FROM country_3nf")

,iso3,name,income_group_code
0,FRA,France,HIC
1,GBR,United Kingdom,HIC
2,IND,India,LMC
3,NGA,Nigeria,LMC


**What you should see:** four countries, each with its name and income-group **code** only. `measure` holds the seven GDP values, one per country and year, and nothing else.

### 4.6 Check: nothing was lost

Normalising must not lose information. A **join** puts the pieces back together (joins are covered in the lecture and the seminar). If the normalised tables are right, joining them reproduces every row and value of the original:

In [41]:
# 🟢 Rebuild the original columns from the normalised tables
rebuild_sql = """
SELECT m.iso3,
       m.year,
       c.name   AS country_name,
       g.code   AS income_group_code,
       g.label  AS income_group_label,
       m.gdp_pc_usd
FROM measure AS m
JOIN country_3nf  AS c ON m.iso3 = c.iso3
JOIN income_group AS g ON c.income_group_code = g.code
ORDER BY m.iso3, m.year;
"""
rebuilt = show(rebuild_sql)

original = show("""
SELECT iso3, year, country_name, income_group_code, income_group_label, gdp_pc_usd
FROM survey_raw
ORDER BY iso3, year;
""")

print("Rows in the original:", len(original))
print("Rows rebuilt:        ", len(rebuilt))
print("Identical:", rebuilt.equals(original))
rebuilt

Rows in the original: 7
Rows rebuilt:         7
Identical: True


,iso3,year,country_name,income_group_code,income_group_label,gdp_pc_usd
0,FRA,2021,France,HIC,High income,43700.0
1,FRA,2022,France,HIC,High income,44400.0
2,GBR,2021,United Kingdom,HIC,High income,46500.0
3,GBR,2022,United Kingdom,HIC,High income,48900.0
4,IND,2021,India,LMC,Lower middle income,2250.0
5,IND,2022,India,LMC,Lower middle income,2410.0
6,NGA,2022,Nigeria,LMC,Lower middle income,2160.0


**What you should see:** 7 rows in both, and `Identical: True`. Every fact in the original table can be recovered from the normalised tables, but each is now **stored once**.

**Reading the join:** `FROM measure AS m` gives the table a short name, `m`; `JOIN country_3nf AS c ON m.iso3 = c.iso3` matches each measurement to its country; the second `JOIN` matches each country to its income group. `m.iso3` means "the `iso3` column of `m`".

And the anomalies are gone:

* **Update:** a country's name is in one row of `country_3nf`; change it once.
* **Insertion:** you can add Spain to `country_3nf` with no measurement yet.
* **Deletion:** deleting Nigeria's only `measure` row leaves Nigeria in `country_3nf`.

### 4.7 The cost: more joins

Normalised tables need **joins** to answer many questions, and joins take time on very large tables. That is why analytical copies are sometimes **denormalised on purpose** (lecture: the cost of denormalisation). The rule of thumb: **normalise the tables you write to; denormalise, deliberately, the copies you analyse.**

### 🔵 Exercise 4: normalise a table

Here is a table of courses taught at a school, keyed by `(course, student)`:

In [ ]:
# 🟢 The table to normalise
con.execute("""
CREATE TABLE enrolment_raw (
    student            TEXT,
    course             TEXT,
    instructor         TEXT,
    instructor_office  TEXT,
    PRIMARY KEY (student, course)
);
""")
enrolments = [
    ("Ana",  "ST445", "Dr Dias",  "COL.7.14"),
    ("Ana",  "ST447", "Dr Wong",  "COL.6.02"),
    ("Ben",  "ST445", "Dr Dias",  "COL.7.14"),
    ("Chen", "ST447", "Dr Wong",  "COL.6.02"),
]
con.executemany("INSERT INTO enrolment_raw VALUES (?, ?, ?, ?)", enrolments)
con.commit()
show("SELECT * FROM enrolment_raw")

✍️ **First, in words.** Which column breaks **2NF**, and which breaks **3NF**? (Hint: the key is `(student, course)`. Which columns depend only on `course`? Which depends on the instructor, not on the course?)

*Your answer:*

🔵 **Then in SQL.** Create three tables, and fill them with `INSERT INTO ... SELECT DISTINCT`:

* `instructor(name PRIMARY KEY, office)`
* `course(code PRIMARY KEY, instructor REFERENCES instructor(name))`
* `enrolment(student, course REFERENCES course(code), PRIMARY KEY (student, course))`

Create them **in that order**: a foreign key can only refer to a table that already exists.

In [ ]:
# Your code here

In [ ]:
# Check for exercise 4
counts = {}
for table in ["instructor", "course", "enrolment"]:
    # table names cannot be ? placeholders; these are fixed names written by us, not user input
    query = "SELECT COUNT(*) AS n FROM " + table
    result = show(query)
    counts[table] = result["n"][0]
check(counts == {"instructor": 2, "course": 2, "enrolment": 4},
      f"Expected 2 instructors, 2 courses and 4 enrolments; found {counts}.")

rebuilt = show("""
SELECT e.student, e.course, c.instructor, i.office AS instructor_office
FROM enrolment AS e
JOIN course AS c ON e.course = c.code
JOIN instructor AS i ON c.instructor = i.name
ORDER BY e.student, e.course;
""")
original = show("SELECT * FROM enrolment_raw ORDER BY student, course")
check(len(rebuilt) == 4 and list(rebuilt["instructor_office"]) == list(original["instructor_office"]),
      "Joining the three tables should rebuild the original four rows.")

**Answer to the ✍️ question.** `instructor` and `instructor_office` depend only on `course`, which is **part** of the key `(student, course)`: that breaks 2NF, so they move to a `course` table. In that table, `instructor_office` depends on `instructor`, a non-key column, not on the course: that breaks 3NF, so it moves to an `instructor` table.

---


---
## 5. Final challenge: from an API to a database

Everything so far used data typed into the notebook. Real projects **collect** data first. This challenge joins Week 2 and this week into one small pipeline:

| Step | What you do | Week |
|---|---|---|
| 1. **Collect** | request population figures from the World Bank API | Week 2 |
| 2. **Prepare** | turn the JSON records into rows: one tuple per country and year | Weeks 2 and 3 |
| 3. **Design** | create a table with a primary key, a foreign key and constraints | this week |
| 4. **Store** | insert the rows with placeholders, in a transaction | this week |
| 5. **Verify** | check the row count, and query the stored data with a join | this week |

The countries are the five in your `country` table, so the foreign key from section 2 will check every row the API sends back.

### Step 1: collect  🟢

The World Bank API (Week 2) needs no key. The URL names the countries (separated by `;`) and the indicator (`SP.POP.TOTL`, total population); the parameters ask for JSON, the years 2018 to 2022, and up to 100 records per page.

We read the country codes **from our own database**, so the request matches what is stored. If the API cannot be reached, the cell uses a small saved sample with the same shape, so you can still complete the challenge.

In [ ]:
# 🟢 Step 1: collect population data for the countries in our database
import requests

rows_from_db = con.execute("SELECT iso3 FROM country ORDER BY iso3").fetchall()
codes = []
for row in rows_from_db:
    codes.append(row[0])
codes_text = ";".join(codes)                 # e.g. "FRA;GBR;IND;NGA;QAT"
print("Countries:", codes_text)

url = f"https://api.worldbank.org/v2/country/{codes_text}/indicator/SP.POP.TOTL"
params = {"format": "json", "date": "2018:2022", "per_page": 100}

try:
    response = requests.get(url, params=params, timeout=30)
    print("Status code:", response.status_code)          # check the status before the body (Week 2)
    response.raise_for_status()
    body = response.json()
    metadata = body[0]
    records = body[1]
    source = "World Bank API"
except Exception as err:
    print("The API could not be reached:", type(err).__name__, "- using the saved sample instead")
    metadata = {"total": 10}
    records = [
        {"countryiso3code": "FRA", "date": "2022", "value": 68170228},
        {"countryiso3code": "FRA", "date": "2021", "value": 67764304},
        {"countryiso3code": "GBR", "date": "2022", "value": 66971395},
        {"countryiso3code": "GBR", "date": "2021", "value": 66744000},
        {"countryiso3code": "IND", "date": "2022", "value": 1425423212},
        {"countryiso3code": "IND", "date": "2021", "value": 1414203896},
        {"countryiso3code": "NGA", "date": "2022", "value": 223150896},
        {"countryiso3code": "NGA", "date": "2021", "value": 218529286},
        {"countryiso3code": "QAT", "date": "2022", "value": 2695122},
        {"countryiso3code": "QAT", "date": "2021", "value": 2688235},
    ]
    source = "saved sample"

print("Records received:", len(records), "| the API says there are:", metadata["total"])
print("The first record:")
pprint_record = records[0]
for key in ["countryiso3code", "date", "value"]:
    print("   ", key, "=", repr(pprint_record[key]))

**What you should see:** the five country codes, `Status code: 200`, and **25 records** (5 countries × 5 years), matching the API's `total`. The first record shows the three fields we need: `countryiso3code` (text), `date` (the year, **as text**: Week 2) and `value` (the population, a whole number, or `None` if not yet published).

(If the sample was used, you will see 10 records instead: the steps that follow work the same.)

### Step 2: prepare  🟡

A database row is a **tuple**, so turn each JSON record into one: `(iso3, year, population)`. Two details from Weeks 2 and 3:

* `date` is text: convert it to a whole number with `int(...)`.
* `value` can be `None`: skip those records, and count them, rather than storing an empty population.

Fill the gaps.

In [ ]:
# 🟡 Step 2: one tuple per record
rows = []
skipped = 0

for record in records:
    population = record["value"]
    if population is None:
        skipped = skipped + 1
        continue                                  # go straight to the next record
    iso3 = record["____"]
    year = ____(record["date"])
    rows.append((iso3, year, population))

print(len(rows), "rows ready;", skipped, "records skipped (no value)")
print("First row:", rows[0])

**What you should see:** 25 rows ready (10 with the sample) and probably 0 skipped, then a first row such as `('FRA', 2022, 68170228)`: text, a whole number, a whole number.

**`continue`** jumps to the next record in the loop, so the lines after it are not run for a record without a value.

### Step 3: design the table  🔵

Create a table called **`population`** that meets these requirements. Each one is something you have done in section 2:

| Column | Type | Rules |
|---|---|---|
| `iso3` | text | required; a **foreign key** to `country(iso3)` |
| `year` | whole number | required; between 1960 and 2030 |
| `population` | whole number | required; greater than 0 |
| `source` | text | where the data came from; default `'World Bank API'` |

and a **composite primary key** on `(iso3, year)`: each country appears once per year.

In [ ]:
# Your code here: CREATE TABLE population (...)

In [ ]:
# Check for step 3: the design
info = show("PRAGMA table_info(population)")
pk_columns = list(info.loc[info["pk"] > 0, "name"])
foreign_keys = show("PRAGMA foreign_key_list(population)")
check(sorted(pk_columns) == ["iso3", "year"], f"The primary key should be (iso3, year); found {pk_columns}.")
check(len(foreign_keys) == 1 and foreign_keys["table"][0] == "country", "iso3 should REFERENCE country(iso3).")

**What the check reads:** `PRAGMA table_info` lists each column with a `pk` number (its position in the primary key, or 0); `PRAGMA foreign_key_list` lists the table's foreign keys and the tables they refer to.

### Step 4: store  🟡

Insert all the rows with **one** `executemany`, using `?` placeholders, inside a `with con:` block so the insert is a **transaction**: if any row breaks a rule, none is stored. Also record the source.

In [ ]:
# 🟡 Step 4: insert every row in one transaction
insert_sql = "INSERT INTO population (iso3, year, population, source) VALUES (?, ?, ?, ?)"

rows_with_source = []
for iso3, year, value in rows:
    rows_with_source.append((iso3, year, value, source))

with con:
    con.____(insert_sql, ____)

print("Stored.")

**What happened:** every tuple was inserted, and `with con:` committed them together. The loop line `for iso3, year, value in rows:` **unpacks** each three-item tuple into three names, so we can build a four-item tuple with the source added.

### Step 5: verify  🔵

Never assume the pipeline worked: check it, as in Week 4.

1. Store in `n_stored` the number of rows in `population` (a `SELECT COUNT(*)` query), and compare it with `len(rows)`.
2. Store in `latest` a DataFrame with each country's **name** (from `country`) and its **population in 2022**, largest first. You need a **join**: `population` gives the figure, `country` gives the name, and they share `iso3`.

In [ ]:
# Your code here

In [ ]:
# Check for step 5
check(n_stored == len(rows), "The number of rows stored should equal the number prepared.")
check(list(latest.columns) == ["name", "population"] and latest["name"][0] == "India",
      "Join population to country on iso3, keep 2022, and sort by population, largest first.")

**What you should see:** the same number of rows prepared and stored, then five countries from India (about 1.4 billion) down to Qatar (about 2.7 million), each with the name taken from `country` through the join.

### ✍️ Reflect

1. The foreign key was checked for every row you inserted. What would have happened if the API had also returned a country that is **not** in your `country` table, say Spain? (Try it: add `"ESP"` to `codes_text` in step 1 and run steps 1 to 4 again.) What would you change in step 2 to handle it?
2. Run step 4 a **second** time. What happens, and which constraint causes it? How would you change the pipeline so it can be re-run safely? (Hint: SQLite has `INSERT OR REPLACE`; or delete the year's rows first, in the same transaction.)

*Your answers:*

**You have built the whole pipeline in about forty lines:** collect from an API, check the response, prepare typed rows, design a table whose constraints reject bad data, store in a transaction, and verify with a count and a join. Your project's data pipeline follows the same steps, at a larger scale.

---

## 6. Closing the connection, and what to do next

Close the connection when you have finished. The database file `week5.db` keeps everything you committed: download it from the Files panel and open it in any SQLite tool (for example *DB Browser for SQLite*, sqlitebrowser.org) to look around.

In [ ]:
# 🟢 Close the connection
con.close()
print("Closed. Everything you committed is saved in", DB_FILE)

### Summary

| You learned | The key syntax |
|---|---|
| Create a table, with types | `CREATE TABLE {table} ({column} {type}, ...);` with `INTEGER`, `REAL`, `TEXT`, `BLOB`; `STRICT` to enforce types |
| Keys and constraints | `PRIMARY KEY`, composite `PRIMARY KEY (a, b)`, `REFERENCES {table}({column})`, `NOT NULL`, `UNIQUE`, `CHECK`, `DEFAULT`; `PRAGMA foreign_keys = ON` |
| Create | `INSERT INTO {table} ({columns}) VALUES (?, ?, ...)`, with `execute` or `executemany` |
| Read | `SELECT ... FROM ... WHERE ... GROUP BY ... HAVING ... ORDER BY ... LIMIT ...`; `IS NULL` for missing values |
| Update and delete | `UPDATE ... SET ... WHERE ...`; `DELETE FROM ... WHERE ...`; look first; check `rowcount` |
| Safety | `?` placeholders, never f-strings; transactions with `with con:` |
| Normalise | 1NF one value per cell; 2NF the whole key; 3NF nothing but the key; check with a join that nothing was lost |

**References.** Codd, E. F. (1970), "A relational model of data for large shared data banks", *Communications of the ACM*, 13(6);

Kent, W. (1983), "A simple guide to five normal forms in relational database theory", *Communications of the ACM*, 26(2);

Ramakrishnan, R. and Gehrke, J. (2003), *Database Management Systems*, 3rd ed., chapter 3; SQLite documentation, sqlite.org/lang.html; Python documentation, "sqlite3".